# Data Quality Audit
Checks across tables before cleaning. Reads from `olist.db`.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../olist.db")

tables = [
    "orders", "order_items", "payments", "reviews", "customers",
    "products", "sellers", "category_translation", "geolocation",
]
data = {t: pd.read_sql_query(f"SELECT * FROM {t}", conn) for t in tables}

In [2]:
for name, df in data.items():
    print(name, len(df))

orders 99441
order_items 112650
payments 103886
reviews 99224
customers 99441
products 32951
sellers 3095
category_translation 71
geolocation 1000163


All tables match the row counts from the load script.

In [3]:
orders = data["orders"]
items = data["order_items"]
payments = data["payments"]
reviews = data["reviews"]
products = data["products"]

## Referential integrity

In [4]:
orders_without_items = set(orders["order_id"]) - set(items["order_id"])
len(orders_without_items)

775

Same as during exploration: 775 orders have no item rows.

In [5]:
orders_without_payments = set(orders["order_id"]) - set(payments["order_id"])
len(orders_without_payments)

1

In [6]:
orders[orders["order_id"].isin(orders_without_payments)]["order_status"].value_counts()

order_status
delivered    1
Name: count, dtype: int64

In [7]:
weird_order = list(orders_without_payments)[0]
orders[orders["order_id"] == weird_order]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04 00:00:00


In [8]:
reviews[reviews["order_id"] == weird_order][["review_score", "review_comment_message", "review_creation_date"]]

,review_score,review_comment_message,review_creation_date
37547,1,nao recebi o produto e nem resposta da empresa,2016-10-06 00:00:00


One delivered order has no payment row at all. The customer's review says they never received the product (score 1), but the order is marked delivered 5 weeks after the estimate. Status and payments are not fully trustworthy for this row. Exclude from payment analysis, keep for the anomaly list.

In [9]:
orders_without_reviews = set(orders["order_id"]) - set(reviews["order_id"])
len(orders_without_reviews)

768

Same as during exploration: 768 orders have no reviews.

In [10]:
orders["order_id"].is_unique

True

Primary key is unique.

## Date logic

In [11]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Date columns now have the correct format.

In [12]:
orders["delivery_days"] = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
orders["days_vs_estimate"] = (orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days

orders[["delivery_days", "days_vs_estimate"]].describe()

,delivery_days,days_vs_estimate
count,96476.000000,96476.000000
mean,12.094086,-11.876881
std,9.551746,10.183854
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


On average, orders arrive about 12 days before the estimated date (median 12 days early). Total delivery time averages about 12 days from purchase to arrival. Worst case was 209 days total, 188 days past the estimate.

In [13]:
orders.loc[orders["delivery_days"].idxmax(), ["delivery_days", "days_vs_estimate"]]

delivery_days       209.0
days_vs_estimate    181.0
Name: 19590, dtype: float64

The longest order (209 days) was 181 days late, so the 209 and 188 maxes are from different orders.

In [14]:
delivered = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].notna()
]
late = delivered["order_delivered_customer_date"].dt.date > delivered["order_estimated_delivery_date"].dt.date
late.mean().round(4).item()

0.0677

About 6.8% of delivered orders arrive later than estimated. Most orders arrive early.

## Value sanity

In [15]:
reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

Scores run 1 to 5, median 5: most customers rate high, so the distribution is top-heavy.

In [16]:
(reviews["review_score"] < 1).sum(), (reviews["review_score"] > 5).sum()

(np.int64(0), np.int64(0))

All scores are within the 1 to 5 range.

In [17]:
payments["payment_value"].describe()

count    103886.000000
mean        154.100380
std         217.494064
min           0.000000
25%          56.790000
50%         100.000000
75%         171.837500
max       13664.080000
Name: payment_value, dtype: float64

In [18]:
(payments["payment_value"] <= 0).sum()

np.int64(9)

In [19]:
(payments["payment_value"] < 0).sum()

np.int64(0)

In [20]:
payments[payments["payment_value"] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


No negative payments, but 9 rows are 0: voucher and not_defined type, not failed cards.

In [21]:
payments["payment_installments"].describe()

count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

Median 1 installment - most people pay in one go. Max 24 - matches the usual Brazilian card cap.

In [22]:
payments["payment_installments"].value_counts().sort_index().head()

payment_installments
0        2
1    52546
2    12413
3    10461
4     7098
Name: count, dtype: int64

In [23]:
payments[payments["payment_installments"] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


2 orders have 0 payment installments.

In [24]:
payments[
    (payments["order_id"] == "744bade1fcf9ff3f31d860ace076d422") |
    (payments["order_id"] == "1a57108394169c0b47d8f876acc9ba2d")
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


Each of these orders has only payment row with payment_sequential = 2; the first payment is missing.

In [25]:
(products["product_weight_g"] == 0).sum()

np.int64(4)

In [26]:
products[products["product_weight_g"] == 0]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


4 products have weight 0 grams but still have dimensions. All in same category (bed bath table).

## Summary

Issues found and planned handling:

| Issue | Where | Severity | Planned fix |
|---|---|---|---|
| Dates stored as text | orders, items, reviews | High | Convert to datetime in cleaning |
| 8 delivered orders missing delivery timestamp | orders | Medium | Keep rows, exclude from delay metrics |
| 1 delivered order with no payment row | payments vs orders | Low | Exclude from payment analysis, keep in anomaly list |
| 2 orders missing first payment row (`payment_sequential` starts at 2, installments 0) | payments | Low | Same, exclude from installment stats |
| 9 payments with value 0 | payments | Low | Keep (voucher/`not_defined`), note in docs |
| Column misspelled: `product_name_lenght`, `product_description_lenght` | products | Medium | Rename to `*_length` |
| 610 products missing all descriptive metadata | products | Low | Keep rows, `null` means unknown |
| 2 products missing weight/dimensions, 4 weigh 0g | products | Low | Treat 0g as `null` |
| 2 categories with no English translation | products vs translation | Low | Keep Portuguese, show English where joined |
| 547 orders with multiple review rows; 789 review IDs span 2 orders | reviews | High | Pick latest review per order (or aggregate) in analysis |
| 775 orders with no items, 768 with no reviews | cross-table | Info | Expected (canceled/early); handle in joins, don't drop |
| Zip prefixes stored as int64 | customers, sellers, geo | Low | Cast to text only when joining on zip |

No rows are deleted in cleaning except where a rule above says exclude from a specific metric.